# Entity Resolution POC — Results Visualization

Loads `results/master_results.csv` and generates:
- Recall@1 heatmap (model × bucket)
- Recall@K curves (K=1,5,10) per model
- BM25 delta plot (all models vs BM25 baseline)
- Memory vs quality scatter
- Latency vs Recall@1 scatter (Pareto frontier)
- Dimensionality ablation curve (exp007)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Config
RESULTS_CSV = Path('../results/master_results.csv')
FIGURES_DIR = Path('../results/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.family'] = 'monospace'
sns.set_theme(style='whitegrid', palette='muted')

print(f'Loading results from {RESULTS_CSV}')
df = pd.read_csv(RESULTS_CSV)
print(f'Loaded {len(df)} rows, {df["experiment_id"].nunique()} experiments')
df.head()

## 1. Recall@1 Heatmap — Model × Bucket

In [ ]:
# Filter to primary experiments (768-dim FP32, 1M index)
primary = df[
    (df['dims'] == 768) & 
    (df['quantization'] == 'fp32') & 
    (df['index_size'] == 1_000_000)
].copy()

# Pivot for heatmap
bucket_order = ['pristine', 'missing_firstname', 'missing_email_company', 
                'typo_name', 'domain_mismatch', 'swapped_attributes']

pivot = primary.pivot_table(
    index='model', 
    columns='bucket', 
    values='recall_at_1'
)[bucket_order]

# Sort by mean recall
pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(
    pivot, 
    annot=True, 
    fmt='.3f', 
    cmap='YlOrRd',
    vmin=0.0, 
    vmax=1.0,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Recall@1 by Model and Corruption Bucket\n(768-dim FP32, 1M index)', 
             fontsize=14, pad=12)
ax.set_xlabel('Corruption Bucket', labelpad=10)
ax.set_ylabel('Model', labelpad=10)
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'recall_at_1_heatmap.png', bbox_inches='tight')
plt.show()
print('Saved to results/figures/recall_at_1_heatmap.png')

## 2. Delta vs BM25 Baseline

In [ ]:
# Get BM25 baseline numbers
bm25 = primary[primary['model'] == 'bm25'].set_index('bucket')['recall_at_1']
non_bm25 = primary[primary['model'] != 'bm25'].copy()

# Compute delta vs BM25
non_bm25['delta_vs_bm25'] = non_bm25.apply(
    lambda row: row['recall_at_1'] - bm25.get(row['bucket'], np.nan), axis=1
)

delta_pivot = non_bm25.pivot_table(
    index='model', 
    columns='bucket', 
    values='delta_vs_bm25'
)[bucket_order]
delta_pivot = delta_pivot.loc[delta_pivot.mean(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(
    delta_pivot,
    annot=True,
    fmt='+.3f',
    cmap='RdYlGn',
    center=0,
    vmin=-0.3,
    vmax=0.5,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Recall@1 Delta vs BM25 Baseline\n(Green=better than BM25, Red=worse)', 
             fontsize=14, pad=12)
ax.set_xlabel('Corruption Bucket', labelpad=10)
ax.set_ylabel('Model', labelpad=10)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'delta_vs_bm25.png', bbox_inches='tight')
plt.show()

## 3. Recall@K Curves — Primary Models

In [ ]:
# Show Recall@1, @5, @10 for each model across buckets
models_to_show = ['bm25', 'nomic_v15', 'nomic_v15_finetuned_pipe', 'nomic_v15_finetuned_kv']
colors = {'bm25': '#e41a1c', 'nomic_v15': '#ff7f00', 
          'nomic_v15_finetuned_pipe': '#4daf4a', 'nomic_v15_finetuned_kv': '#377eb8'}

fig, axes = plt.subplots(2, 3, figsize=(16, 10), sharey=True)
axes = axes.flatten()

for i, bucket in enumerate(bucket_order):
    ax = axes[i]
    bucket_data = primary[
        (primary['bucket'] == bucket) & 
        (primary['model'].isin(models_to_show))
    ]
    
    k_values = [1, 5, 10]
    for _, row in bucket_data.iterrows():
        model = row['model']
        recalls = [row['recall_at_1'], row['recall_at_5'], row['recall_at_10']]
        label = model.replace('nomic_v15', 'nomic').replace('_finetuned', '_ft')
        color = colors.get(model, 'gray')
        ax.plot(k_values, recalls, 'o-', label=label, color=color, linewidth=2, markersize=6)
    
    ax.set_title(bucket.replace('_', ' ').title(), fontsize=11)
    ax.set_xlabel('K')
    ax.set_ylabel('Recall@K')
    ax.set_xticks([1, 5, 10])
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.4)

fig.suptitle('Recall@K Curves by Bucket and Model', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'recall_at_k_curves.png', bbox_inches='tight')
plt.show()

## 4. Dimensionality Ablation (Experiment 007)

In [ ]:
# Experiment 007: Recall@1 vs embedding dims for fine-tuned nomic pipe
exp007 = df[
    (df['experiment_id'] == '007') & 
    (df['model'] == 'nomic_v15_finetuned_pipe') &
    (df['index_size'] == 1_000_000)
].copy()

# Memory calculation
N_RECORDS_MILLIONS = 500  # Extrapolate to 500M
exp007['ram_gb_500m_fp32'] = exp007['dims'] * 4 * N_RECORDS_MILLIONS * 1e6 / 1e9
exp007['ram_gb_500m_binary'] = exp007['dims'] / 8 * N_RECORDS_MILLIONS * 1e6 / 1e9

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Recall@1 vs dims (per bucket)
ax = axes[0]
for bucket in ['pristine', 'typo_name', 'missing_email_company']:
    bucket_data = exp007[(exp007['bucket'] == bucket) & (exp007['quantization'] == 'fp32')]
    bucket_data = bucket_data.sort_values('dims')
    ax.plot(bucket_data['dims'], bucket_data['recall_at_1'], 'o-', 
            label=bucket.replace('_', ' '), linewidth=2)
ax.set_xlabel('Embedding Dimensions')
ax.set_ylabel('Recall@1')
ax.set_title('Quality vs Dimensionality (FP32)\nFine-tuned Nomic v1.5, Pipe Format')
ax.set_xscale('log', base=2)
ax.set_xticks([64, 128, 256, 512, 768])
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.legend()
ax.grid(True, alpha=0.4)

# Right: Quality vs Memory (500M records extrapolation)
ax = axes[1]
pristine_fp32 = exp007[(exp007['bucket'] == 'pristine') & (exp007['quantization'] == 'fp32')].sort_values('dims')
pristine_bin = exp007[(exp007['bucket'] == 'pristine') & (exp007['quantization'] == 'binary')].sort_values('dims')

ax.scatter(pristine_fp32['ram_gb_500m_fp32'], pristine_fp32['recall_at_1'], 
           s=100, label='FP32', zorder=5, color='steelblue')
ax.scatter(pristine_bin['ram_gb_500m_binary'], pristine_bin['recall_at_1'], 
           s=100, label='Binary', zorder=5, color='darkorange', marker='^')

for _, row in pristine_fp32.iterrows():
    ax.annotate(f"{int(row['dims'])}D", 
               (row['ram_gb_500m_fp32'], row['recall_at_1']),
               textcoords='offset points', xytext=(5, 5), fontsize=8)

ax.axvline(x=32, color='red', linestyle='--', alpha=0.5, label='32GB budget')
ax.set_xlabel('Index RAM (GB) at 500M records')
ax.set_ylabel('Recall@1 (Pristine)')
ax.set_title('Memory vs Quality Tradeoff\n(Extrapolated to 500M records)')
ax.set_xscale('log')
ax.legend()
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dimensionality_ablation.png', bbox_inches='tight')
plt.show()

## 5. Latency vs Quality Pareto Frontier

In [ ]:
# Each point = one model configuration
# X = p99 latency (ms), Y = mean Recall@1 across all buckets
pareto_data = primary.copy()
pareto_summary = pareto_data.groupby(['model', 'dims', 'quantization']).agg(
    mean_recall_at_1=('recall_at_1', 'mean'),
    mean_recall_at_10=('recall_at_10', 'mean'),
    p99_latency_ms=('p99_latency_ms', 'mean'),
).reset_index()

fig, ax = plt.subplots(figsize=(10, 6))

for _, row in pareto_summary.iterrows():
    label = f"{row['model']}\n{row['dims']}D {row['quantization']}"
    ax.scatter(row['p99_latency_ms'], row['mean_recall_at_1'], s=100, zorder=5)
    ax.annotate(label, (row['p99_latency_ms'], row['mean_recall_at_1']),
               textcoords='offset points', xytext=(5, 5), fontsize=7,
               bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

ax.set_xlabel('p99 Query Latency (ms)')
ax.set_ylabel('Mean Recall@1 (across 6 buckets)')
ax.set_title('Latency vs Quality — All Model Configurations\n(Better = upper-left corner)')
ax.axvline(x=100, color='red', linestyle='--', alpha=0.5, label='100ms p99 target')
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'latency_vs_quality.png', bbox_inches='tight')
plt.show()

## 6. Summary Table — Final Rankings

In [ ]:
# Final summary: mean Recall@1 across all buckets, ranked
summary = primary.groupby('model').agg(
    mean_R1=('recall_at_1', 'mean'),
    mean_R10=('recall_at_10', 'mean'),
    mean_MRR=('mrr_at_10', 'mean'),
    mean_p99_ms=('p99_latency_ms', 'mean'),
).sort_values('mean_R1', ascending=False).round(4)

# BM25 delta for each model
bm25_mean_r1 = summary.loc['bm25', 'mean_R1'] if 'bm25' in summary.index else 0
summary['delta_vs_bm25_R1'] = (summary['mean_R1'] - bm25_mean_r1).round(4)

print('=== FINAL RESULTS SUMMARY ===')
print(f'BM25 baseline mean Recall@1: {bm25_mean_r1:.4f}')
print()
print(summary.to_string())

# Save as formatted HTML for reports
summary.to_html(FIGURES_DIR / 'summary_table.html')
summary.to_csv(FIGURES_DIR / 'summary_table.csv')
print('\nSaved summary table to results/figures/')